# Red Cedar — Quick Start

The basic Red Cedar workflow in four steps:
1. Train a quantum model on a dataset
2. Look at how training went
3. Test the model on held-out data
4. See which input features the model relies on

All computation happens on the server; this notebook sends requests and plots the results. To explore a dataset before training (PCA, mutual information, qubit estimates) see `02_dataset_analysis.ipynb`. All SDK functions are described in the redcedar README.

In [ ]:
import redcedar
import matplotlib.pyplot as plt

# Connect to the server
# Reads REDCEDAR_URL and REDCEDAR_API_KEY from environment, or pass explicitly:
# client = redcedar.connect(url="http://localhost:8002", api_key="your-key")
client = redcedar.connect()
print(f"Red Cedar SDK v{redcedar.__version__}")
print(f"Server: {client.url}")
print(f"Status: {client.status()}")

dataset = "wdbc"   # Wisconsin breast cancer data; client.datasets.list() shows all datasets

## 1. Train a model

Training runs layer by layer: each layer adds a set of operators to the circuit and starts from the parameters the previous layer found. Here we use 6 qubits and 2 layers, which trains in about a minute. `seed` fixes the starting parameters, so the run can be repeated exactly.

At the end of each layer the server tests the model on held-out data; those results come back in `result["validation"]`.

In [ ]:
result = client.training.run(dataset, verbose=True, settings={
    "num_qubits": 6,
    "min_layers": 1,
    "max_layers": 2,
    "seed": 123,
})

train_id = result["train_id"]
print(f"\nModel ID: {train_id}")
print(f"Training time: {result['elapsed_seconds']}s")
for layer, v in result["validation"].items():
    print(f"Layer {layer}: test accuracy {v['test_accuracy']:.1%}, test loss {v['test_loss']:.4f}")

## 2. How training went

`result["history"]` holds the training loss and accuracy after every parameter update, for each layer. The dashed lines mark where a new layer starts.

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
start = 0
for layer, points in result["history"].items():
    steps = range(start, start + len(points))
    ax_loss.plot(steps, [p["train_loss"] for p in points], label=f"layer {layer}")
    ax_acc.plot(steps, [p["train_accuracy"] for p in points], label=f"layer {layer}")
    if start:
        ax_loss.axvline(start, color="gray", ls="--", lw=0.8)
        ax_acc.axvline(start, color="gray", ls="--", lw=0.8)
    start += len(points)

ax_loss.set(xlabel="Parameter update", ylabel="Training loss", title="Training loss")
ax_acc.set(xlabel="Parameter update", ylabel="Training accuracy", title="Training accuracy")
for ax in (ax_loss, ax_acc):
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Test the model

Run the trained model (by default, its last layer) on the held-out test data.

In [ ]:
inference = client.inference.run(train_id, verbose=True)

print(f"\nOverall accuracy: {inference['overall_accuracy']:.1%} on {inference['test_samples']} test samples")
for cls in inference["per_class"]:
    print(f"  Class {cls['class']}: accuracy {cls['accuracy']:.1%} ({cls['samples']} samples)")

classes = [str(c["class"]) for c in inference["per_class"]]
plt.figure(figsize=(5, 3))
plt.bar(classes, [c["accuracy"] for c in inference["per_class"]])
plt.axhline(inference["overall_accuracy"], color="gray", ls="--", label="overall")
plt.ylim(0, 1)
plt.xlabel("Class")
plt.ylabel("Test accuracy")
plt.legend()
plt.show()

## 4. Which features the model relies on

Explainable AI maps each data qubit back to the input feature it encodes and measures how much that qubit influences the prediction.

In [ ]:
explanation = client.explainable.run(train_id, verbose=True)

qubits = explanation["qubit_map"]
for q in qubits:
    print(f"  Qubit {q['qubit']}: {q['feature_name']} (importance {q['importance']:.1%})")

plt.figure(figsize=(6, 3))
plt.barh([f"q{q['qubit']}: {q['feature_name']}" for q in qubits], [q["importance"] for q in qubits])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.show()

## Next steps

- `02_dataset_analysis.ipynb`: explore a dataset before training, or analyze your own CSV file
- `03_training_settings.ipynb`: settings you can change, one section per setting
- `04_training_in_depth.ipynb`: layers, test accuracy over the course of training, choosing the data, reusing models, stopping a job